<a href="https://colab.research.google.com/github/eman3304/project1/blob/main/project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import sqlite3
from google.colab import files
from IPython.display import display

print("✅ Libraries imported successfully")


print("📂 Upload the CLEANED dataset from Project 1")
print("Example: Cleaned_Dataset.xlsx")

uploaded = files.upload()

file_name = next(iter(uploaded))

print(f"\n✅ File uploaded: {file_name}")



df = pd.read_excel(file_name)

print("\n" + "="*70)
print("DATASET LOADED")
print("="*70)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

display(df.head())

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
)

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

if "Date" in df.columns:
    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")



connection = sqlite3.connect("decodelabs_project3.db")

df.to_sql(
    "sales_data",
    connection,
    if_exists="replace",
    index=False
)

print("\n✅ SQL table created successfully: sales_data")


def run_sql(query, title=None):

    if title:
        print("\n" + "="*70)
        print(title)
        print("="*70)

    result = pd.read_sql_query(
        query,
        connection
    )

    display(result)

    return result



run_sql(
    """
    PRAGMA table_info(sales_data);
    """,
    "1. TABLE STRUCTURE"
)



run_sql(
    """
    SELECT *
    FROM sales_data
    LIMIT 10;
    """,
    "2. SELECT - FIRST 10 ROWS"
)



available_columns = set(df.columns)

if {"OrderID", "CustomerID", "Product", "TotalPrice"}.issubset(
    available_columns
):

    run_sql(
        """
        SELECT
            OrderID,
            CustomerID,
            Product,
            TotalPrice
        FROM sales_data
        LIMIT 20;
        """,
        "3. SELECT SPECIFIC COLUMNS"
    )



if "TotalPrice" in available_columns:

    run_sql(
        """
        SELECT *
        FROM sales_data
        WHERE TotalPrice > 100
        ORDER BY TotalPrice DESC;
        """,
        "4. WHERE - ORDERS ABOVE 100"
    )


if {"Product", "TotalPrice"}.issubset(available_columns):

    run_sql(
        """
        SELECT
            Product,
            TotalPrice
        FROM sales_data
        WHERE TotalPrice > 50
        ORDER BY TotalPrice DESC;
        """,
        "5. WHERE + ORDER BY"
    )



if "OrderID" in available_columns:

    run_sql(
        """
        SELECT
            COUNT(*) AS Total_Orders
        FROM sales_data;
        """,
        "6. COUNT - TOTAL ORDERS"
    )


if "CustomerID" in available_columns:

    run_sql(
        """
        SELECT
            COUNT(DISTINCT CustomerID) AS Unique_Customers
        FROM sales_data;
        """,
        "7. COUNT DISTINCT - UNIQUE CUSTOMERS"
    )


if "TotalPrice" in available_columns:

    run_sql(
        """
        SELECT
            ROUND(SUM(TotalPrice), 2) AS Total_Revenue
        FROM sales_data;
        """,
        "8. SUM - TOTAL REVENUE"
    )



if "TotalPrice" in available_columns:

    run_sql(
        """
        SELECT
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data;
        """,
        "9. AVG - AVERAGE ORDER VALUE"
    )


if "TotalPrice" in available_columns:

    run_sql(
        """
        SELECT
            ROUND(MIN(TotalPrice), 2) AS Minimum_Order,
            ROUND(MAX(TotalPrice), 2) AS Maximum_Order
        FROM sales_data;
        """,
        "10. MIN AND MAX ORDER VALUE"
    )



if {"Product", "OrderID", "Quantity", "TotalPrice"}.issubset(
    available_columns
):

    product_analysis = run_sql(
        """
        SELECT
            Product,
            COUNT(*) AS Orders,
            SUM(Quantity) AS Total_Quantity,
            ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        GROUP BY Product
        ORDER BY Total_Revenue DESC;
        """,
        "11. GROUP BY - PRODUCT PERFORMANCE"
    )


if {"OrderStatus", "OrderID", "TotalPrice"}.issubset(
    available_columns
):

    status_analysis = run_sql(
        """
        SELECT
            OrderStatus,
            COUNT(*) AS Orders,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        GROUP BY OrderStatus
        ORDER BY Orders DESC;
        """,
        "12. GROUP BY - ORDER STATUS"
    )


if {"PaymentMethod", "OrderID", "TotalPrice"}.issubset(
    available_columns
):

    payment_analysis = run_sql(
        """
        SELECT
            PaymentMethod,
            COUNT(*) AS Orders,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        GROUP BY PaymentMethod
        ORDER BY Revenue DESC;
        """,
        "13. GROUP BY - PAYMENT METHOD"
    )


if {"ReferralSource", "OrderID", "TotalPrice"}.issubset(
    available_columns
):

    referral_analysis = run_sql(
        """
        SELECT
            ReferralSource,
            COUNT(*) AS Orders,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        GROUP BY ReferralSource
        ORDER BY Revenue DESC;
        """,
        "14. GROUP BY - REFERRAL SOURCE"
    )


if {"CouponCode", "OrderID", "TotalPrice"}.issubset(
    available_columns
):

    coupon_analysis = run_sql(
        """
        SELECT
            CouponCode,
            COUNT(*) AS Orders,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        GROUP BY CouponCode
        ORDER BY Orders DESC;
        """,
        "15. GROUP BY - COUPON PERFORMANCE"
    )


if {"Product", "TotalPrice"}.issubset(
    available_columns
):

    run_sql(
        """
        SELECT
            Product,
            ROUND(SUM(TotalPrice), 2) AS Total_Revenue
        FROM sales_data
        GROUP BY Product
        HAVING SUM(TotalPrice) > 5000
        ORDER BY Total_Revenue DESC;
        """,
        "16. HAVING - PRODUCTS ABOVE REVENUE THRESHOLD"
    )



if {"CustomerID", "OrderID", "TotalPrice"}.issubset(
    available_columns
):

    customer_analysis = run_sql(
        """
        SELECT
            CustomerID,
            COUNT(*) AS Number_of_Orders,
            ROUND(SUM(TotalPrice), 2) AS Total_Spending,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        GROUP BY CustomerID
        ORDER BY Total_Spending DESC
        LIMIT 20;
        """,
        "17. TOP 20 CUSTOMERS BY SPENDING"
    )



if {"CustomerID", "OrderID"}.issubset(
    available_columns
):

    run_sql(
        """
        SELECT
            CustomerID,
            COUNT(*) AS Number_of_Orders
        FROM sales_data
        GROUP BY CustomerID
        HAVING COUNT(*) > 1
        ORDER BY Number_of_Orders DESC;
        """,
        "18. REPEAT CUSTOMERS"
    )



if {"Date", "TotalPrice"}.issubset(
    available_columns
):

    monthly_analysis = run_sql(
        """
        SELECT
            strftime('%Y-%m', Date) AS Month,
            COUNT(*) AS Orders,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value
        FROM sales_data
        WHERE Date IS NOT NULL
        GROUP BY strftime('%Y-%m', Date)
        ORDER BY Month;
        """,
        "19. MONTHLY SALES ANALYSIS"
    )



if {"Date", "TotalPrice"}.issubset(
    available_columns
):

    run_sql(
        """
        SELECT
            strftime('%Y-%m', Date) AS Month,
            ROUND(SUM(TotalPrice), 2) AS Revenue
        FROM sales_data
        WHERE Date IS NOT NULL
        GROUP BY strftime('%Y-%m', Date)
        ORDER BY Revenue DESC
        LIMIT 1;
        """,
        "20. HIGHEST REVENUE MONTH"
    )


if {"Product", "Quantity"}.issubset(
    available_columns
):

    run_sql(
        """
        SELECT
            Product,
            SUM(Quantity) AS Total_Quantity
        FROM sales_data
        GROUP BY Product
        ORDER BY Total_Quantity DESC
        LIMIT 10;
        """,
        "21. TOP PRODUCTS BY QUANTITY SOLD"
    )


if {"OrderID", "Product", "TotalPrice"}.issubset(
    available_columns
):

    run_sql(
        """
        SELECT
            OrderID,
            Product,
            TotalPrice
        FROM sales_data
        ORDER BY TotalPrice DESC
        LIMIT 10;
        """,
        "22. TOP 10 HIGHEST-VALUE ORDERS"
    )


if "TotalPrice" in available_columns:

    run_sql(
        """
        SELECT
            CASE
                WHEN TotalPrice < 50 THEN 'Low Value'
                WHEN TotalPrice < 100 THEN 'Medium Value'
                ELSE 'High Value'
            END AS Order_Value_Category,

            COUNT(*) AS Orders,

            ROUND(SUM(TotalPrice), 2) AS Revenue,

            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value

        FROM sales_data

        GROUP BY
            CASE
                WHEN TotalPrice < 50 THEN 'Low Value'
                WHEN TotalPrice < 100 THEN 'Medium Value'
                ELSE 'High Value'
            END

        ORDER BY Revenue DESC;
        """,
        "23. ORDER VALUE SEGMENTATION"
    )


if {
    "Product",
    "PaymentMethod",
    "TotalPrice"
}.issubset(available_columns):

    run_sql(
        """
        SELECT
            Product,
            PaymentMethod,
            COUNT(*) AS Orders,
            ROUND(SUM(TotalPrice), 2) AS Revenue,
            ROUND(AVG(TotalPrice), 2) AS Average_Order_Value

        FROM sales_data

        GROUP BY
            Product,
            PaymentMethod

        ORDER BY
            Revenue DESC;
        """,
        "24. PRODUCT + PAYMENT METHOD ANALYSIS"
    )


print("\n" + "="*70)
print("25. SQL BUSINESS SUMMARY")
print("="*70)

summary_query = """
SELECT
    COUNT(*) AS Total_Orders,
    COUNT(DISTINCT CustomerID) AS Unique_Customers,
    ROUND(SUM(TotalPrice), 2) AS Total_Revenue,
    ROUND(AVG(TotalPrice), 2) AS Average_Order_Value,
    ROUND(MIN(TotalPrice), 2) AS Minimum_Order,
    ROUND(MAX(TotalPrice), 2) AS Maximum_Order
FROM sales_data;
"""

if {
    "CustomerID",
    "TotalPrice"
}.issubset(available_columns):

    business_summary = run_sql(
        summary_query,
        "BUSINESS SUMMARY"
    )


output_file = "DecodeLabs_Project_3_SQL_Analysis.xlsx"

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="SQL_Data",
        index=False
    )

    if "product_analysis" in locals():
        product_analysis.to_excel(
            writer,
            sheet_name="Product_Analysis",
            index=False
        )

    if "status_analysis" in locals():
        status_analysis.to_excel(
            writer,
            sheet_name="Order_Status",
            index=False
        )

    if "payment_analysis" in locals():
        payment_analysis.to_excel(
            writer,
            sheet_name="Payment_Method",
            index=False
        )

    if "referral_analysis" in locals():
        referral_analysis.to_excel(
            writer,
            sheet_name="Referral_Source",
            index=False
        )

    if "coupon_analysis" in locals():
        coupon_analysis.to_excel(
            writer,
            sheet_name="Coupon_Analysis",
            index=False
        )

    if "customer_analysis" in locals():
        customer_analysis.to_excel(
            writer,
            sheet_name="Top_Customers",
            index=False
        )

    if "monthly_analysis" in locals():
        monthly_analysis.to_excel(
            writer,
            sheet_name="Monthly_Analysis",
            index=False
        )

    if "business_summary" in locals():
        business_summary.to_excel(
            writer,
            sheet_name="Business_Summary",
            index=False
        )


connection.close()


print("\n" + "="*70)
print("PROJECT 3 COMPLETED")
print("="*70)

print(f"✅ SQL analysis file created: {output_file}")

files.download(output_file)

print("\n🎉 DecodeLabs Project 3 - SQL Data Analysis completed successfully!")

✅ Libraries imported successfully
📂 Upload the CLEANED dataset from Project 1
Example: Cleaned_Dataset.xlsx


Saving Cleaned_Dataset.xlsx to Cleaned_Dataset.xlsx

✅ File uploaded: Cleaned_Dataset.xlsx

DATASET LOADED
Rows    : 1,200
Columns : 14


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04



✅ SQL table created successfully: sales_data

1. TABLE STRUCTURE


,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,TEXT,0,None,0
1,1,Date,TEXT,0,None,0
2,2,CustomerID,TEXT,0,None,0
3,3,Product,TEXT,0,None,0
4,4,Quantity,INTEGER,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,ShippingAddress,TEXT,0,None,0
7,7,PaymentMethod,TEXT,0,None,0
8,8,OrderStatus,TEXT,0,None,0
9,9,TrackingNumber,TEXT,0,None,0


...

,Total_Orders,Unique_Customers,Total_Revenue,Average_Order_Value,Minimum_Order,Maximum_Order
0,1200,1189,1264761.96,1053.97,11.39,3456.4



PROJECT 3 COMPLETED
✅ SQL analysis file created: DecodeLabs_Project_3_SQL_Analysis.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 DecodeLabs Project 3 - SQL Data Analysis completed successfully!
